# RCKangaroo build + smoke test (Colab T4)

Verifies the GPU solver builds and recovers a known key on a small
synthetic puzzle. Run this once before pointing at the real target.

**Why RCKangaroo?** Puzzle #135 needs a >125-bit interval — JLP caps at
125-bit, K-256 hangs on Colab T4 in our setup. RCKangaroo (RetiredC) is
the canonical >125-bit solver: 130 ⭐, GPL-3.0, supports up to 170-bit
ranges, builds for sm_75 (T4) by default, uses the SOTA equivalence-class
negation map (~1.15·√n vs JLP's ~2.08·√n — ~1.8× faster).

**Prereq:** Runtime → Change runtime type → T4 GPU.


In [ ]:
# Sanity: confirm a T4 is attached.
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader


In [ ]:
# Session setup: wipe stale state and clone the project fresh.
# Colab sessions expire; /content/ is wiped each cold start. This cell is
# idempotent — re-run anytime to reset to a known-good state.
import os, sys, shutil, subprocess

REPO_URL    = "https://github.com/anevolbap/bitcoin-prize.git"
PROJECT_DIR = "/content/bitcoin-prize"
KANGAROO_DIR = "/content/Kangaroo"
WORK_DIR    = "/content/work"

# Wipe any stale clones / build dirs from a prior session.
for d in (PROJECT_DIR, KANGAROO_DIR, "/content/Kangaroo-256"):
    if os.path.isdir(d):
        shutil.rmtree(d)
        print(f"removed {d}")

subprocess.run(
    ["git", "clone", "--depth", "1", REPO_URL, PROJECT_DIR],
    check=True,
)
os.makedirs(WORK_DIR, exist_ok=True)
sys.path.insert(0, PROJECT_DIR)
print(f"OK: project at {PROJECT_DIR}")


In [ ]:
# Clone + build RetiredC/RCKangaroo. We detect nvcc dynamically because
# the Makefile hardcodes /usr/local/cuda-12.0/bin/nvcc which doesn't exist
# on Colab (Colab ships CUDA at /usr/local/cuda → versioned dir, e.g. -12.5).
import shutil

if not os.path.isdir(KANGAROO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/RetiredC/RCKangaroo.git", KANGAROO_DIR],
        check=True,
    )

nvcc_path = shutil.which("nvcc")
if not nvcc_path:
    raise RuntimeError("nvcc not on PATH; check Colab CUDA install")
cuda_root = os.path.dirname(os.path.dirname(nvcc_path))  # .../bin/nvcc → ...
print(f"nvcc:      {nvcc_path}\ncuda_root: {cuda_root}")

subprocess.run(["make", "clean"], cwd=KANGAROO_DIR, check=False,
               capture_output=True)
build = subprocess.run(
    ["make",
     f"CUDA_PATH={cuda_root}",
     f"NVCC={nvcc_path}"],
    cwd=KANGAROO_DIR, capture_output=True, text=True,
)
print(build.stdout[-2000:])
if build.returncode != 0:
    print("STDERR:", build.stderr[-2000:])
    raise RuntimeError(f"build failed (rc={build.returncode})")

binary = os.path.join(KANGAROO_DIR, "rckangaroo")
assert os.path.isfile(binary), f"binary missing after build: {binary}"
print("built:", binary)


In [ ]:
# Generate a small synthetic puzzle with a known answer.
# 60 bits → ~2 G ops → ~4 s on T4 at ~600 MK/s.
# Solve time scales as sqrt(N): each +2 bits ≈ 2× wall time.
PUZZLE_PATH = f"{WORK_DIR}/puzzle.txt"
BITS = 60
SEED = 42

gen = subprocess.run(
    ["python3", "scripts/make_synthetic_puzzle.py",
     "--bits", str(BITS), "--seed", str(SEED), "--out", PUZZLE_PATH],
    cwd=PROJECT_DIR, capture_output=True, text=True, check=True,
)
print(gen.stderr.strip())
print("---")
print(open(PUZZLE_PATH).read())


In [ ]:
# Run RCKangaroo against the synthetic puzzle.
# RCKangaroo uses CLI args (not input file): -range, -start, -pubkey.
# It writes the result to RESULTS.TXT in CWD; we run from WORK_DIR and
# wipe any stale file first.
import time, json

RESULT_PATH = os.path.join(WORK_DIR, "RESULTS.TXT")
if os.path.exists(RESULT_PATH):
    os.remove(RESULT_PATH)

with open(PUZZLE_PATH + ".json") as f:
    manifest = json.load(f)

# RCKangaroo's -range = log2(interval width) = bits - 1 (puzzle #N → N-1).
rc_range = manifest["bits"] - 1
start_hex = manifest["k1_hex"]
pubkey = manifest["pubkey_hex"]
DP_BITS = 14   # RCKangaroo's minimum; fine for narrow smoke intervals

print(f"BITS={BITS}, -range {rc_range}, -dp {DP_BITS}, expecting d = 0x{manifest['d_hex']}")

t0 = time.monotonic()
solve = subprocess.run(
    [binary, "-gpu", "0",
     "-dp", str(DP_BITS),
     "-range", str(rc_range),
     "-start", start_hex,
     "-pubkey", pubkey],
    cwd=WORK_DIR, capture_output=True, text=True, timeout=120,
)
elapsed = time.monotonic() - t0
print(solve.stdout[-2500:])
print(f"--- elapsed {elapsed:.2f}s, rc={solve.returncode}")


In [ ]:
# Verify: read RESULTS.TXT (RCKangaroo writes the key there on success).
# Falls back to scanning stdout for any 30+ hex char string if the file
# format surprises us.
import re

if os.path.exists(RESULT_PATH):
    out = open(RESULT_PATH).read()
    print(f"--- {RESULT_PATH} ---")
    print(out)
else:
    print(f"{RESULT_PATH} missing; scanning stdout")
    out = solve.stdout

# Try common label patterns first; fall back to any long hex token.
patterns = [
    r"Priv(?:ate)?\s*[Kk]ey\s*:\s*(?:0x)?([0-9a-fA-F]+)",
    r"Priv\s*:\s*(?:0x)?([0-9a-fA-F]+)",
    r"\b([0-9a-fA-F]{30,})\b",  # last resort: any sufficiently long hex
]
recovered = None
for pat in patterns:
    m = re.search(pat, out)
    if m:
        recovered = m.group(1).lower().lstrip("0") or "0"
        break

assert recovered is not None, f"no key parsed from output:\n{out[-500:]}"
expected = manifest["d_hex"].lstrip("0") or "0"
assert recovered == expected, f"key mismatch: got {recovered}, expected {expected}"
print(f"PASS — recovered d = 0x{recovered}")


## What this proves

- The Linux/CUDA build path on Colab T4 works for RCKangaroo.
- RCKangaroo recovers a known key on the synthetic puzzle.
- The toolchain — clone → build → puzzle gen → solve → verify — is reproducible.

**Next:** build the production-target notebook (`solver.ipynb`) on top of
this: drop the synthetic puzzle, point at puzzle #135 constants from
`kangaroo/puzzle_135.py`, add Drive-mounted work-file checkpointing
every ~20 min, idle-clean exit on session timeout.
